# Task 5 — Regional Node Network

Select regional hubs from the CoStar candidate pool using a capacity-aware set-cover MIP, then define the regional hub network topology using geographic and freight-interaction criteria.

**Notebook scope:** Tasks 5.1 through 5.6 (H/Z construction → MIP → analysis → network links → figures).

---

## Task 5.1 — Candidate Set H and Z Construction

Build the filtered hub candidate set **H** and the feasibility set **Z**.

### Steps
1. **H construction** — apply adaptive size floor; sparse-region fallback from broader pool
2. **Road accessibility β gate** — compute $d_h^{road}$ for each candidate; exclude the 10% most remote
3. **Z construction** — all $(h, r)$ pairs where Euclidean distance ≤ 150 miles (241 402 m)

### Key parameters
| Symbol | Value | Meaning |
| ------ | ----- | ------- |
| base floor | 100 000 sqft | Minimum `usable_available_space_sf` from primary pool |
| fallback floor | 50 000 sqft | Minimum sqft from supplementary pool for sparse regions |
| $\beta$ | 90th pct | Maximum $d_h^{road}$ allowed in H |
| 150 mi | 241 402 m | Maximum hub-to-region centroid distance for Z |
| 75 mi | 120 701 m | Z-neighbor radius for sparse-region check |

In [1]:
# ── Imports and paths ────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import geopandas as gpd
import shapely
from shapely import STRtree
from pyproj import Transformer
from pathlib import Path

# Locate project root (contains both Data/ and Doc/), regardless of CWD
ROOT = Path(".").resolve()
for _ in range(4):
    if (ROOT / "Data").exists() and (ROOT / "Doc").exists():
        break
    ROOT = ROOT.parent

DATA_T3    = ROOT / "Data/Task3"
DATA_T4    = ROOT / "Data/Task4/processed"
DATA_T5    = ROOT / "Data/Task5"
CACHE      = DATA_T5 / "cache"
CACHE.mkdir(parents=True, exist_ok=True)

# Constants
BASE_FLOOR     = 100_000          # sqft — base filter from primary pool
FALLBACK_FLOOR =  50_000          # sqft — fallback for sparse regions
SPARSE_N       = 2                # min in-region candidates before fallback check
SPARSE_RADIUS  = 75 * 1609.34    # 75 miles in metres
MAX_DIST_M     = 150 * 1609.34   # 150 miles in metres  (= 241 402 m)
BETA_PCTILE    = 90               # road accessibility gate percentile
EXEMPT_REGIONS = {43, 19}        # confirmed external Z coverage; no fallback needed

# CRS transformer: WGS-84 → EPSG:9311 (equal-area, metres)
PROJ = Transformer.from_crs("EPSG:4326", "EPSG:9311", always_xy=True)

print("Imports OK")
print(f"ROOT: {ROOT}")

Imports OK
ROOT: /home/hty/Documents/Working Projects/ISYE6339_Case2


---

## 5.1.1 — H Construction: Adaptive Size Floor

**Base filter** — retain candidates from `primary_regional_hub_candidates.csv` with:
$$s_h \geq 100{,}000 \text{ sqft}$$

Because Task 4 already enforced $s_h \geq 200{,}000$ sqft on the primary pool, this filter effectively keeps every row and merely documents the floor used in Task 5.

**Sparse-region fallback** — for region $r$ with fewer than 2 in-region candidates *and* fewer than 2 H-candidates within 75 miles of its centroid, pull additional candidates from the broader `preprocessed_capacity_location.csv` pool where $s_h \geq 50{,}000$ sqft.

Regions 43 (ME) and 19 (rural PA/WV) are exempt: both have ≥ 160 H-candidates within 150 miles via neighboring regions, providing sufficient Z coverage without fallback.

In [2]:
# ── Load primary candidate pool ───────────────────────────────────────────────
primary = pd.read_csv(DATA_T4 / "primary_regional_hub_candidates.csv")
print(f"Primary pool  : {len(primary):>6,} rows")
print(f"Columns       : {list(primary.columns)}")
print(f"sqft range    : {primary['usable_available_space_sf'].min():,.0f} – "
      f"{primary['usable_available_space_sf'].max():,.0f}")

Primary pool  :  1,862 rows
Columns       : ['candidate_id', 'source_state', 'facility_name', 'city', 'county_fips', 'county_name', 'region_id', 'latitude', 'longitude', 'secondary_type', 'building_class', 'building_status', 'year_built', 'usable_available_space_sf', 'number_loading_docks', 'availability_class', 'is_directly_usable_by_status', 'meets_min_rba_200k', 'is_primary_regional_hub_candidate']
sqft range    : 200,000 – 500,000


In [3]:
# ── Apply base filter ─────────────────────────────────────────────────────────
H = primary[primary['usable_available_space_sf'] >= BASE_FLOOR].copy().reset_index(drop=True)
print(f"After base filter (≥{BASE_FLOOR:,} sqft): {len(H):,} candidates")

# Project to EPSG:9311
H_x, H_y = PROJ.transform(H['longitude'].values, H['latitude'].values)
H['x_m'] = H_x
H['y_m'] = H_y

# Per-region candidate count (before fallback)
region_counts_pre = H.groupby('region_id').size().rename('n_base')
print(f"\nRegion candidate count (base):")
print(region_counts_pre.describe().round(1))

After base filter (≥100,000 sqft): 1,862 candidates

Region candidate count (base):
count     50.0
mean      37.2
std       26.1
min        7.0
25%       14.2
50%       30.5
75%       52.5
max      121.0
Name: n_base, dtype: float64


In [4]:
# ── Sparse-region fallback ────────────────────────────────────────────────────
region_metrics = pd.read_csv(DATA_T3 / "outputs/region_metrics.csv")
region_metrics = region_metrics.sort_values('region_id').reset_index(drop=True)

H_coords = H[['x_m', 'y_m']].values

sparse_needs_fallback = []
for _, row in region_metrics.iterrows():
    r_id = int(row['region_id'])
    if r_id in EXEMPT_REGIONS:
        continue
    in_region = (H['region_id'] == r_id).sum()
    if in_region < SPARSE_N:
        # Count any H candidate within 75 miles of this region centroid
        dists = np.sqrt(
            (H_coords[:, 0] - row['centroid_x_m'])**2 +
            (H_coords[:, 1] - row['centroid_y_m'])**2
        )
        nearby = int((dists <= SPARSE_RADIUS).sum())
        if nearby < SPARSE_N:
            sparse_needs_fallback.append(r_id)
            print(f"  Region {r_id:2d}: {in_region} in-region, {nearby} within 75 mi → FALLBACK")

if not sparse_needs_fallback:
    print("No regions need fallback candidates.")
else:
    print(f"\nRegions needing fallback: {sparse_needs_fallback}")

No regions need fallback candidates.


In [5]:
# ── Pull fallback candidates (if any) ────────────────────────────────────────
if sparse_needs_fallback:
    preprocessed = pd.read_csv(DATA_T4 / "preprocessed_capacity_location.csv")
    fallback = preprocessed[
        preprocessed['region_id'].isin(sparse_needs_fallback) &
        (preprocessed['usable_available_space_sf'] >= FALLBACK_FLOOR) &
        (~preprocessed['candidate_id'].isin(H['candidate_id']))
    ].copy()
    if len(fallback) > 0:
        fb_x, fb_y = PROJ.transform(fallback['longitude'].values, fallback['latitude'].values)
        fallback['x_m'] = fb_x
        fallback['y_m'] = fb_y
        H = pd.concat([H, fallback], ignore_index=True)
        print(f"Added {len(fallback)} fallback candidates for regions {sparse_needs_fallback}")
    else:
        print("Fallback pool had no matching candidates.")

H = H.reset_index(drop=True)
H['h_idx'] = H.index          # integer index used in Z pairs

print(f"\n|H| after base filter + fallback = {len(H):,}")
print(f"sqft range: {H['usable_available_space_sf'].min():,.0f} – "
      f"{H['usable_available_space_sf'].max():,.0f}")


|H| after base filter + fallback = 1,862
sqft range: 200,000 – 500,000


---

## 5.1.2 — Road Accessibility Pre-filter (β Gate)

For each candidate $h \in H$, compute the minimum Euclidean distance in EPSG:9311 to any US interstate segment:

$$d_h^{road} = \min_{s \in \mathcal{S}_{int}} \|\mathbf{x}_h - \text{proj}(\mathbf{x}_h, s)\|_2$$

where $\mathcal{S}_{int}$ is the set of US interstate segments (COUNTRY = 2, CLASS = 1) from `North_American_Roads.shp`, projected to EPSG:9311.

The road accessibility threshold is set at:
$$\beta = \text{pct}_{90}(d_h^{road} \mid h \in H)$$

Candidates with $d_h^{road} > \beta$ are excluded from H as a hard pre-filter (Constraint 5 in the MIP), keeping the model size smaller than if β were implemented as a MIP variable.

In [6]:
# ── Load US interstates (or from cache) ───────────────────────────────────────
int_cache = CACHE / "us_interstates.parquet"

if int_cache.exists():
    interstates = gpd.read_parquet(int_cache)
    print(f"Loaded US interstates from cache: {len(interstates):,} segments")
else:
    ROADS_SHP = DATA_T3 / "raw/roads/North_American_Roads.shp"
    print(f"Reading {ROADS_SHP} ...")
    roads = gpd.read_file(ROADS_SHP)
    print(f"Roads columns: {list(roads.columns)}")
    print(f"COUNTRY dtype: {roads['COUNTRY'].dtype}, sample: {roads['COUNTRY'].unique()[:5]}")
    print(f"CLASS   dtype: {roads['CLASS'].dtype},   sample: {roads['CLASS'].unique()[:5]}")

    # Coerce to int if stored as float/string
    roads['COUNTRY'] = pd.to_numeric(roads['COUNTRY'], errors='coerce')
    roads['CLASS']   = pd.to_numeric(roads['CLASS'],   errors='coerce')

    mask = (roads['COUNTRY'] == 2) & (roads['CLASS'] == 1)
    interstates = roads[mask].to_crs("EPSG:9311")
    interstates.to_parquet(int_cache)
    print(f"Cached {len(interstates):,} US interstate segments → {int_cache}")

Loaded US interstates from cache: 100,009 segments


In [7]:
# ── Compute d_h^road via STRtree nearest-neighbour ────────────────────────────
d_road_cache = CACHE / "d_road.npy"
H_idx_cache  = CACHE / "H_pre_beta.parquet"

if d_road_cache.exists() and H_idx_cache.exists():
    d_road = np.load(d_road_cache)
    H_pre  = pd.read_parquet(H_idx_cache)
    # Reload H from pre-beta snapshot so h_idx aligns
    H      = H_pre.copy()
    print(f"Loaded d_road from cache: {len(d_road):,} values")
else:
    int_geoms  = interstates.geometry.values
    strtree    = STRtree(int_geoms)

    H_points   = shapely.points(H['x_m'].values, H['y_m'].values)
    nearest_ix = strtree.nearest(H_points)
    d_road     = shapely.distance(H_points, int_geoms[nearest_ix])

    np.save(d_road_cache, d_road)
    H['d_road_m']     = d_road
    H['d_road_miles'] = d_road / 1609.34
    H.to_parquet(H_idx_cache, index=False)
    print(f"Computed and cached d_road for {len(d_road):,} candidates")

H['d_road_m']     = d_road
H['d_road_miles'] = d_road / 1609.34

print(f"\nd_road distribution (metres):")
print(pd.Series(d_road).describe().round(1))

Loaded d_road from cache: 1,862 values

d_road distribution (metres):
count      1862.0
mean       5585.6
std       11573.3
min          44.0
25%         763.5
50%        1953.7
75%        4929.9
max      127582.7
dtype: float64


In [8]:
# ── Apply β gate ─────────────────────────────────────────────────────────────
beta_m     = float(np.percentile(d_road, BETA_PCTILE))
beta_miles = beta_m / 1609.34

n_before = len(H)
H = H[H['d_road_m'] <= beta_m].copy().reset_index(drop=True)
H['h_idx'] = H.index
n_excluded = n_before - len(H)

print(f"β = {BETA_PCTILE}th percentile d_road = {beta_m:,.0f} m  ({beta_miles:.2f} miles)")
print(f"Excluded  : {n_excluded:,} candidates (d_road > β)")
print(f"|H| final : {len(H):,} candidates")

# Per-region breakdown after β
region_counts_post = H.groupby('region_id').size().rename('n_post_beta')
print(f"\nRegion candidate count (post β-gate):")
print(region_counts_post.describe().round(1))

β = 90th percentile d_road = 12,831 m  (7.97 miles)
Excluded  : 187 candidates (d_road > β)
|H| final : 1,675 candidates

Region candidate count (post β-gate):
count     49.0
mean      34.2
std       25.9
min        5.0
25%       13.0
50%       25.0
75%       51.0
max      112.0
Name: n_post_beta, dtype: float64


---

## 5.1.3 — Z Construction

The feasibility set $Z$ contains all hub–region pairs $(h, r)$ where the Euclidean distance from hub $h$ to region $r$'s centroid (both in EPSG:9311) is at most 150 miles:

$$Z = \left\{ (h, r) \in H \times R \;\middle|\; \|\mathbf{x}_h - \boldsymbol{\mu}_r\|_2 \leq 241{,}402 \text{ m} \right\}$$

where $\boldsymbol{\mu}_r = (\text{centroid\_x\_m}_r, \text{centroid\_y\_m}_r)$ from `region_metrics.csv`.

The pairwise distance matrix $D \in \mathbb{R}^{|H| \times 50}$ is computed once via NumPy broadcasting and cached for reuse in Task 5.2.

In [9]:
# ── Build Z (h, r) feasibility pairs ─────────────────────────────────────────
Z_cache      = CACHE / "Z_pairs.npy"
dist_hr_path = CACHE / "dist_hr.npy"

# Region centroids ordered by region_id (0..49)
region_metrics = region_metrics.sort_values('region_id').reset_index(drop=True)
R_ids    = region_metrics['region_id'].values            # shape (50,)
R_coords = region_metrics[['centroid_x_m', 'centroid_y_m']].values  # (50, 2)

# Hub coordinates
H_coords = H[['x_m', 'y_m']].values                     # (|H|, 2)

# Pairwise Euclidean distance via broadcasting: (|H|, 50)
diffs   = H_coords[:, np.newaxis, :] - R_coords[np.newaxis, :, :]   # (|H|, 50, 2)
dist_hr = np.sqrt((diffs ** 2).sum(axis=2))                          # (|H|, 50)

# Z: indices where distance ≤ 150 miles
h_idx_z, r_idx_z = np.where(dist_hr <= MAX_DIST_M)
Z = np.column_stack([h_idx_z, r_idx_z]).astype(np.int32)   # (|Z|, 2)

np.save(dist_hr_path, dist_hr)
np.save(Z_cache, Z)

print(f"|H| = {len(H):,}")
print(f"|R| = {len(R_ids)}")
print(f"|Z| = {len(Z):,} feasible (h, r) pairs")
print(f"Mean Z-pairs per region : {len(Z)/len(R_ids):,.0f}")
print(f"Mean Z-pairs per hub    : {len(Z)/len(H):.1f}")

|H| = 1,675
|R| = 50
|Z| = 26,133 feasible (h, r) pairs
Mean Z-pairs per region : 523
Mean Z-pairs per hub    : 15.6


In [10]:
# ── Verify full region coverage in Z ─────────────────────────────────────────
covered_regions = set(r_idx_z.tolist())
all_region_ix   = set(range(len(R_ids)))
uncovered_ix    = all_region_ix - covered_regions

if uncovered_ix:
    uncovered_ids = [int(R_ids[i]) for i in uncovered_ix]
    print(f"WARNING: {len(uncovered_ix)} region(s) have NO hub in Z: region_ids = {uncovered_ids}")
else:
    print(f"✓ All {len(R_ids)} regions have ≥ 1 hub in Z")

# Per-region Z hub count
z_per_region = pd.Series(r_idx_z).value_counts().sort_index()
print(f"\nZ-hubs per region  min={z_per_region.min():,}  "
      f"median={z_per_region.median():,.0f}  max={z_per_region.max():,}")

# Special check for exempt regions 43 and 19
for rid in sorted(EXEMPT_REGIONS):
    r_pos = int(np.where(R_ids == rid)[0][0])
    z_count = int((r_idx_z == r_pos).sum())
    print(f"  Region {rid} (exempt) : {z_count} Z-hubs from external candidates")

✓ All 50 regions have ≥ 1 hub in Z

Z-hubs per region  min=44  median=462  max=936
  Region 19 (exempt) : 396 Z-hubs from external candidates
  Region 43 (exempt) : 936 Z-hubs from external candidates


---

## 5.1.4 — Save Outputs and Summary

In [11]:
# ── Persist final H dataframe ─────────────────────────────────────────────────
H_out_path = CACHE / "H_candidates.parquet"
H.to_parquet(H_out_path, index=False)

# Persist β scalar
np.save(CACHE / "beta_m.npy", np.array([beta_m]))

print("Outputs saved:")
print(f"  H_candidates.parquet  → {H_out_path}  ({len(H):,} rows)")
print(f"  Z_pairs.npy           → {CACHE / 'Z_pairs.npy'}  ({len(Z):,} pairs)")
print(f"  dist_hr.npy           → {CACHE / 'dist_hr.npy'}  shape {dist_hr.shape}")
print(f"  beta_m.npy            → {CACHE / 'beta_m.npy'}  β={beta_m:,.0f} m")

# Human-readable H summary
h_summary = H.groupby('region_id').agg(
    n_candidates=('h_idx', 'count'),
    min_sqft=('usable_available_space_sf', 'min'),
    max_sqft=('usable_available_space_sf', 'max'),
    median_d_road_miles=('d_road_miles', 'median')
).reset_index()
h_summary.to_csv(CACHE / "H_region_summary.csv", index=False)
print(f"  H_region_summary.csv  → {CACHE / 'H_region_summary.csv'}")

Outputs saved:
  H_candidates.parquet  → /home/hty/Documents/Working Projects/ISYE6339_Case2/Data/Task5/cache/H_candidates.parquet  (1,675 rows)
  Z_pairs.npy           → /home/hty/Documents/Working Projects/ISYE6339_Case2/Data/Task5/cache/Z_pairs.npy  (26,133 pairs)
  dist_hr.npy           → /home/hty/Documents/Working Projects/ISYE6339_Case2/Data/Task5/cache/dist_hr.npy  shape (1675, 50)
  beta_m.npy            → /home/hty/Documents/Working Projects/ISYE6339_Case2/Data/Task5/cache/beta_m.npy  β=12,831 m
  H_region_summary.csv  → /home/hty/Documents/Working Projects/ISYE6339_Case2/Data/Task5/cache/H_region_summary.csv


---

## 5.1.5 — Sanity Checks

Expected bounds drawn from `Doc/Task.md` and project parameters:

| Check | Bound | Source |
| ----- | ----- | ------ |
| `|H|` size | 500 – 2 000 | After base filter + β removal |
| All regions covered in Z | 50/50 | Constraint 1 feasibility |
| β > 0 | always | 90th percentile of positive distances |
| Z index range | h < |H|, r < 50 | Index integrity |
| d_road ≥ 0 | always | Physical distance |
| sqft ≥ BASE_FLOOR in H | always | H construction guarantee |
| max(dist_hr) for covered pairs ≤ MAX_DIST_M | ≤ 241 402 m | Z construction rule |

In [12]:
# ── Sanity checks ─────────────────────────────────────────────────────────────
errors = []

# 1. H size
if not (500 <= len(H) <= 2000):
    errors.append(f"|H| = {len(H)} outside expected range [500, 2000]")
else:
    print(f"✓ |H| = {len(H):,}  (expected 500–2 000)")

# 2. Full region coverage
if uncovered_ix:
    errors.append(f"Uncovered regions in Z: {[int(R_ids[i]) for i in uncovered_ix]}")
else:
    print(f"✓ All 50 regions covered in Z")

# 3. β positive
if beta_m <= 0:
    errors.append(f"β = {beta_m:.1f} m ≤ 0")
else:
    print(f"✓ β = {beta_m:,.0f} m ({beta_miles:.2f} mi) > 0")

# 4. Z index integrity
if len(Z) > 0:
    if Z[:, 0].max() >= len(H):
        errors.append("Z contains h_idx >= |H|")
    if Z[:, 1].max() >= len(R_ids):
        errors.append("Z contains r_idx >= |R|")
    if not (Z[:, 0].min() >= 0 and Z[:, 1].min() >= 0):
        errors.append("Z contains negative index")
    if not errors:
        print(f"✓ Z index range: h ∈ [0,{Z[:,0].max()}], r ∈ [0,{Z[:,1].max()}]")

# 5. d_road ≥ 0
if (H['d_road_m'] < 0).any():
    errors.append("Negative d_road values in H")
else:
    print(f"✓ All d_road ≥ 0  "
          f"(range: {H['d_road_m'].min():.0f} – {H['d_road_m'].max():,.0f} m)")

# 6. Size floor
if (H['usable_available_space_sf'] < BASE_FLOOR).any():
    errors.append(f"H contains candidates below BASE_FLOOR={BASE_FLOOR:,}")
else:
    print(f"✓ All H candidates have sqft ≥ {BASE_FLOOR:,}  "
          f"(median: {H['usable_available_space_sf'].median():,.0f})")

# 7. Z pair distances all ≤ MAX_DIST_M
if len(Z) > 0:
    z_dists = dist_hr[Z[:, 0], Z[:, 1]]
    if z_dists.max() > MAX_DIST_M + 1:   # +1 m tolerance for float arithmetic
        errors.append(f"Z contains pair with dist {z_dists.max():.0f} m > {MAX_DIST_M:.0f} m")
    else:
        print(f"✓ All Z pair distances ≤ {MAX_DIST_M/1609.34:.0f} mi  "
              f"(max in Z: {z_dists.max()/1609.34:.1f} mi)")

# ── Final verdict ──
print()
if errors:
    for e in errors:
        print(f"  FAIL: {e}")
    raise AssertionError(f"{len(errors)} sanity check(s) failed — see above.")
else:
    print("══════════════════════════════════════")
    print("  All sanity checks PASSED — Task 5.1")
    print("══════════════════════════════════════")

✓ |H| = 1,675  (expected 500–2 000)
✓ All 50 regions covered in Z
✓ β = 12,831 m (7.97 mi) > 0
✓ Z index range: h ∈ [0,1674], r ∈ [0,49]
✓ All d_road ≥ 0  (range: 44 – 12,820 m)
✓ All H candidates have sqft ≥ 100,000  (median: 306,250)
✓ All Z pair distances ≤ 150 mi  (max in Z: 150.0 mi)

══════════════════════════════════════
  All sanity checks PASSED — Task 5.1
══════════════════════════════════════


---

## Milestone — Task 5.1 Complete

| Output | Path | Description |
| ------ | ---- | ----------- |
| `H_candidates.parquet` | `Data/Task5/cache/` | Final filtered hub candidate set with d_road |
| `Z_pairs.npy` | `Data/Task5/cache/` | (h_idx, r_idx) pairs — feasibility set Z |
| `dist_hr.npy` | `Data/Task5/cache/` | Full (|H| × 50) distance matrix for Task 5.2 |
| `beta_m.npy` | `Data/Task5/cache/` | Scalar β value |
| `H_region_summary.csv` | `Data/Task5/cache/` | Per-region candidate count and sqft stats |

**Next:** Task 5.2 — Objective Coefficients and Capacity Parameters (pre-compute ĉ_hr, cap_rhs, Q̄, T̄)

---

## Task 5.2 — Objective Coefficients and Capacity Parameters

Pre-compute all MIP cost coefficients and capacity RHS values outside the solver loop.

### Objective cost coefficient

For each $(h, r) \in Z$, the capacity-adjusted demand-weighted distance is:

$$\hat{c}_{hr} = \left( \sum_{c \in C_r} w_{cr} \cdot d_{hcr} \right) \cdot \left( \frac{\bar{Q}}{s_h} \right)^{0.5}$$

where:
- $w_{cr}$ = county $c$'s 2025 bidirectional throughput (ktons) — the freight-demand weight
- $d_{hcr}$ = Euclidean distance (EPSG:9311 m) from hub $h$ to county $c$ centroid
- $\bar{Q}$ = median $s_h$ across all $h \in H$ — the capacity normalizer
- $s_h$ = `usable_available_space_sf` — hub floor area (sqft)

The $(\bar{Q}/s_h)^{0.5}$ discount keeps geography primary while favouring larger facilities: a hub at the median size gets no adjustment; a hub at 2× the median receives a 29% cost reduction.

### Capacity constraint RHS

$$\text{RHS}_r = \bar{Q} \cdot \frac{T_r}{\bar{T}}$$

where $T_r$ is region $r$'s external throughput (ktons) and $\bar{T}$ is the mean across all 50 regions. This demand-scales the required sqft so high-throughput regions attract proportionally larger (or more) hubs.

In [13]:
# ── 5.2.1  Load Task 5.1 outputs and county data ──────────────────────────────
import geopandas as gpd

# Reload H and Z (in case kernel was restarted)
H   = pd.read_parquet(CACHE / "H_candidates.parquet")
Z   = np.load(CACHE / "Z_pairs.npy")           # (|Z|, 2)  int32  [h_idx, r_idx]

# County layer: centroid coords + throughput weight (w_cr)
counties = gpd.read_file(DATA_T3 / "derived/ne_counties_prepared.gpkg")

# Region assignment: fips → region_id
# counties.fips is object (string); region_assign.fips is int64 → align to string
region_assign = pd.read_csv(DATA_T3 / "outputs/region_assignment.csv")
region_assign['fips'] = region_assign['fips'].astype(str).str.zfill(5)

# Merge region_id onto county layer
counties = counties.merge(
    region_assign[['fips', 'region_id']],
    on='fips', how='left'
)
assert counties['region_id'].notna().all(), "Some counties have no region_id after merge"

# Load corrected county external throughput (hub-facing demand weight, w_cr)
county_ext = pd.read_parquet(DATA_T3 / "derived/county_external_throughput.parquet")
county_ext['fips'] = county_ext['fips'].astype(str).str.zfill(5)
counties = counties.merge(
    county_ext[['fips', 'external_throughput']],
    on='fips', how='left'
)
counties['external_throughput'] = counties['external_throughput'].fillna(0.0)
# Rename to demand_weight for clarity in downstream cells
counties = counties.rename(columns={'external_throughput': 'demand_weight'})

# Region metrics (sorted by region_id → r_idx == region_id)
region_metrics = pd.read_csv(DATA_T3 / "outputs/region_metrics.csv") \
                   .sort_values('region_id').reset_index(drop=True)
R_ids = region_metrics['region_id'].values       # shape (50,)
T_r   = region_metrics['external_throughput_ktons'].values  # shape (50,) hub-facing demand

print(f"H         : {len(H):,} hubs")
print(f"Z         : {len(Z):,} pairs")
print(f"Counties  : {len(counties):,} units  (region_id range {counties['region_id'].min()}–{counties['region_id'].max()})")
print(f"Regions   : {len(R_ids)}")

H         : 1,675 hubs
Z         : 26,133 pairs
Counties  : 434 units  (region_id range 0–49)
Regions   : 50


In [14]:
# ── 5.2.2  Scalar parameters Q̄ and T̄ ─────────────────────────────────────────
s_h   = H['usable_available_space_sf'].values     # shape (|H|,)
Q_bar = float(np.median(s_h))                     # median sqft across H
T_bar = float(T_r.mean())                         # mean regional throughput

print(f"Q̄ (median sqft across H) = {Q_bar:,.0f} sqft")
print(f"T̄ (mean T_r across 50 regions) = {T_bar:,.2f} ktons")
print(f"\nCapacity discount factor range:")
disc_min = (Q_bar / s_h.max()) ** 0.5
disc_max = (Q_bar / s_h.min()) ** 0.5
print(f"  min (largest hub,  s_h={s_h.max():,.0f}): {disc_min:.3f}  ({(1-disc_min)*100:.1f}% discount)")
print(f"  max (smallest hub, s_h={s_h.min():,.0f}): {disc_max:.3f}  ({(disc_max-1)*100:.1f}% premium)")

Q̄ (median sqft across H) = 306,250 sqft
T̄ (mean T_r across 50 regions) = 8,946.88 ktons

Capacity discount factor range:
  min (largest hub,  s_h=500,000): 0.783  (21.7% discount)
  max (smallest hub, s_h=200,000): 1.237  (23.7% premium)


In [15]:
# ── 5.2.3  County centroid arrays by region ───────────────────────────────────
# Build per-region lookup: r_id → (county_coords (n_c,2), county_weights (n_c,))
county_arr = counties[['centroid_x', 'centroid_y', 'demand_weight', 'region_id']].copy()
county_arr = county_arr.rename(columns={'centroid_x': 'cx', 'centroid_y': 'cy',
                                        'demand_weight': 'w'})

# Replace any zero/NaN throughput with a small positive sentinel (avoid zero-weight counties)
county_arr['w'] = county_arr['w'].fillna(0.0).clip(lower=1e-6)

region_counties: dict[int, dict] = {}
for r_id in range(len(R_ids)):
    mask = county_arr['region_id'] == r_id
    sub  = county_arr[mask]
    region_counties[r_id] = {
        'coords':  sub[['cx', 'cy']].values.astype(np.float64),   # (n_c, 2)
        'weights': sub['w'].values.astype(np.float64),             # (n_c,)
    }

n_counties_total = sum(len(v['weights']) for v in region_counties.values())
print(f"County groups built: {len(region_counties)} regions, {n_counties_total} county-slots total")
print(f"Counties per region: min={min(len(v['weights']) for v in region_counties.values())}  "
      f"max={max(len(v['weights']) for v in region_counties.values())}  "
      f"mean={n_counties_total/len(region_counties):.1f}")

County groups built: 50 regions, 434 county-slots total
Counties per region: min=1  max=33  mean=8.7


In [16]:
# ── 5.2.4  Compute ĉ_hr for all (h, r) ∈ Z ───────────────────────────────────
# Strategy: group Z by r_idx, then for each region batch-compute distances
#           from all member hubs to all counties in one vectorised call.
#
# c_hat[z] = (Σ_{c ∈ C_r} w_cr · d_hcr) · (Q̄ / s_h)^0.5
# Shape of c_hat: (|Z|,)  — same ordering as rows of Z_pairs.npy

c_hat_cache = CACHE / "c_hat.npy"

if c_hat_cache.exists():
    c_hat = np.load(c_hat_cache)
    print(f"Loaded c_hat from cache: {len(c_hat):,} values")
else:
    from collections import defaultdict

    H_x  = H['x_m'].values       # (|H|,)
    H_y  = H['y_m'].values       # (|H|,)
    s_h_ = H['usable_available_space_sf'].values  # (|H|,)

    # Group Z rows by r_idx
    z_by_r: dict[int, list] = defaultdict(list)
    for z_pos, (h_pos, r_pos) in enumerate(Z):
        z_by_r[int(r_pos)].append((z_pos, int(h_pos)))

    c_hat = np.empty(len(Z), dtype=np.float64)

    for r_pos, entries in z_by_r.items():
        rc      = region_counties[r_pos]
        c_xy    = rc['coords']          # (n_c, 2)
        c_w     = rc['weights']         # (n_c,)

        z_idxs  = [e[0] for e in entries]
        h_idxs  = [e[1] for e in entries]

        hx = H_x[h_idxs]               # (n_h,)
        hy = H_y[h_idxs]               # (n_h,)
        sh = s_h_[h_idxs]              # (n_h,)

        # Distance from each hub to each county: (n_h, n_c)
        dx     = hx[:, np.newaxis] - c_xy[:, 0][np.newaxis, :]
        dy     = hy[:, np.newaxis] - c_xy[:, 1][np.newaxis, :]
        d_hcr  = np.sqrt(dx**2 + dy**2)

        # Demand-weighted distance sum: (n_h,)
        w_dist = d_hcr @ c_w            # matrix-vector product = Σ_c w_cr · d_hcr

        # Capacity discount: (Q̄ / s_h)^0.5  shape (n_h,)
        cap_disc = np.sqrt(Q_bar / sh)

        c_hat[z_idxs] = w_dist * cap_disc

    np.save(c_hat_cache, c_hat)
    print(f"Computed and cached c_hat: {len(c_hat):,} values")

print(f"\nĉ_hr distribution:")
print(pd.Series(c_hat).describe().apply(lambda x: f"{x:,.0f}"))

Loaded c_hat from cache: 26,133 values

ĉ_hr distribution:
count           26,133
mean     1,193,695,532
std        610,883,907
min         19,622,807
25%        732,677,188
50%      1,157,066,493
75%      1,588,718,244
max      5,282,734,319
dtype: object


In [17]:
# ── 5.2.5  Capacity constraint RHS ────────────────────────────────────────────
# RHS_r = Q̄ × (T_r / T̄)   shape (50,) aligned with R_ids / region_metrics row order
cap_rhs = Q_bar * (T_r / T_bar)      # (50,)

print(f"Capacity RHS distribution (sqft):")
print(pd.Series(cap_rhs, index=R_ids).describe().apply(lambda x: f"{x:,.0f}"))

print(f"\nFeasibility check (single-hub coverage):")
print(f"  max(RHS_r)  = {cap_rhs.max():,.0f} sqft")
print(f"  max(s_h)    = {H['usable_available_space_sf'].max():,.0f} sqft")
single_hub_feasible = cap_rhs.max() <= H['usable_available_space_sf'].max()
status = "OK — every region can be served by a single large hub" if single_hub_feasible \
         else "INFEASIBLE for single hub — two hubs needed for high-demand regions (valid; p_h ≤ 2 allows this)"
print(f"  Status: {status}")

# Save
cap_rhs_path = CACHE / "cap_rhs.npy"
np.save(cap_rhs_path, cap_rhs)
np.save(CACHE / "Q_bar.npy",  np.array([Q_bar]))
np.save(CACHE / "T_bar.npy",  np.array([T_bar]))
print(f"\nSaved: cap_rhs.npy, Q_bar.npy, T_bar.npy → {CACHE}")

Capacity RHS distribution (sqft):
count         50
mean     306,250
std       70,295
min      241,087
25%      270,660
50%      290,042
75%      312,892
max      676,581
dtype: object

Feasibility check (single-hub coverage):
  max(RHS_r)  = 676,581 sqft
  max(s_h)    = 500,000 sqft
  Status: INFEASIBLE for single hub — two hubs needed for high-demand regions (valid; p_h ≤ 2 allows this)

Saved: cap_rhs.npy, Q_bar.npy, T_bar.npy → /home/hty/Documents/Working Projects/ISYE6339_Case2/Data/Task5/cache


---

## 5.2.6 — Sanity Checks

| Check | Bound | Rationale |
| ----- | ----- | --------- |
| `len(c_hat) == len(Z)` | exact | One coefficient per Z pair |
| All `c_hat > 0` | always | Distance × positive weight × positive discount |
| `RHS_r > 0` for all r | always | T_r > 0 for all regions |
| `mean(RHS_r) ≈ Q̄` | within 1 sqft | By construction, mean(T_r/T̄) = 1 |
| `c_hat` finite | always | No NaN/Inf from distance or sqft computations |
| County count per region matches `region_metrics.n_counties` | exact | Merge integrity |

In [18]:
# ── Sanity checks — Task 5.2 ──────────────────────────────────────────────────
errors = []

# 1. c_hat length matches Z
if len(c_hat) != len(Z):
    errors.append(f"len(c_hat)={len(c_hat)} ≠ len(Z)={len(Z)}")
else:
    print(f"✓ len(c_hat) = {len(c_hat):,}  matches |Z|")

# 2. All c_hat > 0
if not (c_hat > 0).all():
    n_bad = (c_hat <= 0).sum()
    errors.append(f"{n_bad} c_hat entries ≤ 0")
else:
    print(f"✓ All c_hat > 0  (min={c_hat.min():,.0f}, max={c_hat.max():,.0f})")

# 3. c_hat all finite
if not np.isfinite(c_hat).all():
    errors.append(f"{(~np.isfinite(c_hat)).sum()} non-finite c_hat values")
else:
    print(f"✓ All c_hat finite")

# 4. RHS_r all positive
if not (cap_rhs > 0).all():
    errors.append(f"{(cap_rhs <= 0).sum()} RHS_r values ≤ 0")
else:
    print(f"✓ All cap_rhs > 0  (min={cap_rhs.min():,.0f}, max={cap_rhs.max():,.0f})")

# 5. mean(RHS_r) ≈ Q̄  (by construction: mean(T_r/T̄) = 1)
mean_rhs = float(cap_rhs.mean())
if abs(mean_rhs - Q_bar) > 1.0:
    errors.append(f"mean(RHS_r)={mean_rhs:,.1f} differs from Q̄={Q_bar:,.0f} by > 1 sqft")
else:
    print(f"✓ mean(cap_rhs) = {mean_rhs:,.1f} ≈ Q̄ = {Q_bar:,.0f}  (diff {abs(mean_rhs-Q_bar):.3f})")

# 6. County count per region matches region_metrics
county_counts_actual = counties.groupby('region_id').size()
county_counts_ref    = region_metrics.set_index('region_id')['n_counties']
mismatch = 0
for r_id in range(len(R_ids)):
    if r_id not in county_counts_actual.index:
        continue
    actual = int(county_counts_actual[r_id])
    ref    = int(county_counts_ref[r_id])
    if actual != ref:
        mismatch += 1
if mismatch:
    errors.append(f"{mismatch} region(s) have county count mismatching region_metrics.n_counties")
else:
    print(f"✓ County counts per region match region_metrics.n_counties (all {len(R_ids)} regions)")

# Final verdict
print()
if errors:
    for e in errors:
        print(f"  FAIL: {e}")
    raise AssertionError(f"{len(errors)} sanity check(s) failed.")
else:
    print("══════════════════════════════════════")
    print("  All sanity checks PASSED — Task 5.2")
    print("══════════════════════════════════════")

✓ len(c_hat) = 26,133  matches |Z|
✓ All c_hat > 0  (min=19,622,807, max=5,282,734,319)
✓ All c_hat finite
✓ All cap_rhs > 0  (min=241,087, max=676,581)
✓ mean(cap_rhs) = 306,250.0 ≈ Q̄ = 306,250  (diff 0.000)
✓ County counts per region match region_metrics.n_counties (all 50 regions)

══════════════════════════════════════
  All sanity checks PASSED — Task 5.2
══════════════════════════════════════


---

## Milestone — Task 5.2 Complete

| Output | Path | Description |
| ------ | ---- | ----------- |
| `c_hat.npy` | `Data/Task5/cache/` | Objective cost coefficients ĉ_hr, shape (25,667,) aligned to Z_pairs |
| `cap_rhs.npy` | `Data/Task5/cache/` | Capacity RHS Q̄·(T_r/T̄), shape (50,) aligned to region_metrics row order |
| `Q_bar.npy` | `Data/Task5/cache/` | Scalar median sqft Q̄ |
| `T_bar.npy` | `Data/Task5/cache/` | Scalar mean regional throughput T̄ |

**Next:** Task 5.3 — MIP Build and Gurobi Solution (define binary variables, assemble constraints 1–4, solve)

---

## Task 5.3 — MIP Build and Gurobi Solution

Build and solve the hub-selection MIP using `gurobipy`.  The optimization logic lives
in `Task5/mip_solver.py` (`RegionalHubMIP` class); this section is the orchestration
and validation layer.

### Formulation recap

$$
\min \sum_{(h,r)\in Z} \hat{c}_{hr} \cdot A_{hr}
$$

| # | Constraint | Rule |
|---|-----------|------|
| 1 | Coverage      | $\sum_{h:(h,r)\in Z} A_{hr} \geq 1 \quad \forall r$ |
| 2 | Concentration | $\sum_{r:(h,r)\in Z} A_{hr} \leq 2 \quad \forall h$ |
| 3 | Link          | $O_h \leq \sum_{r:(h,r)\in Z} A_{hr} \quad \forall h$ |
| 4 | Capacity      | $\sum_{h:(h,r)\in Z} A_{hr} \cdot s_h \geq \bar{Q}(T_r/\bar{T}) \quad \forall r$ |

Constraint 5 (road-accessibility gate) was enforced as a pre-filter in Task 5.1.

### Solver parameters

| Parameter | Value |
|-----------|-------|
| `TimeLimit` | 600 s |
| `MIPGap`    | 1 %  |
| `OutputFlag`| 1 (console log) |
| `concentration_cap` | 2 (p_h) |

In [19]:
# ── 5.3.1  Load all Task 5.1 / 5.2 cached inputs ─────────────────────────────
import sys
import numpy as np
import pandas as pd
from pathlib import Path

# Locate project root
ROOT = Path(".").resolve()
for _ in range(4):
    if (ROOT / "Data").exists() and (ROOT / "Doc").exists():
        break
    ROOT = ROOT.parent

# Add Task5/ to path so we can import mip_solver
TASK5_DIR = ROOT / "Task5"
if str(TASK5_DIR) not in sys.path:
    sys.path.insert(0, str(TASK5_DIR))

from mip_solver import RegionalHubMIP, MIPParameters

DATA_T3 = ROOT / "Data/Task3"
DATA_T5 = ROOT / "Data/Task5"
CACHE   = DATA_T5 / "cache"
OUT_DIR = DATA_T5
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Load cached arrays ────────────────────────────────────────────────────────
H       = pd.read_parquet(CACHE / "H_candidates.parquet")    # 1,675 rows
Z       = np.load(CACHE / "Z_pairs.npy")                     # (25,667, 2)
c_hat   = np.load(CACHE / "c_hat.npy")                       # (25,667,)
cap_rhs = np.load(CACHE / "cap_rhs.npy")                     # (50,)
Q_bar   = float(np.load(CACHE / "Q_bar.npy"))                # scalar
T_bar   = float(np.load(CACHE / "T_bar.npy"))                # scalar

# Region IDs in row order (r_idx == region_id since IDs are 0–49)
region_metrics = (
    pd.read_csv(DATA_T3 / "outputs/region_metrics.csv")
    .sort_values("region_id")
    .reset_index(drop=True)
)
R_ids   = region_metrics["region_id"].values    # (50,)
T_r     = region_metrics["external_throughput_ktons"].values  # (50,) hub-facing demand
s_h     = H["usable_available_space_sf"].values  # (|H|,)
H_arr   = H["h_idx"].values                      # (|H|,)

print(f"|H|     = {len(H):,}")
print(f"|Z|     = {len(Z):,}")
print(f"|R|     = {len(R_ids)}")
print(f"Q̄       = {Q_bar:,.0f} sqft")
print(f"T̄       = {T_bar:,.2f} ktons")
print(f"cap_rhs : min={cap_rhs.min():,.0f}  max={cap_rhs.max():,.0f} sqft")

|H|     = 1,675
|Z|     = 26,133
|R|     = 50
Q̄       = 306,250 sqft
T̄       = 8,946.88 ktons
cap_rhs : min=241,087  max=676,581 sqft


In [20]:
# ── 5.3.2  Build the Gurobi model ─────────────────────────────────────────────
params = MIPParameters(
    time_limit        = 600,   # 10-minute wall-clock limit
    mip_gap           = 0.01,  # stop at 1% optimality gap
    output_flag       = 1,     # print Gurobi log
    concentration_cap = 2,     # each hub serves ≤ 2 regions
    threads           = 0,     # use all available cores
)

solver = RegionalHubMIP(
    H_arr   = H_arr,
    Z       = Z,
    c_hat   = c_hat,
    cap_rhs = cap_rhs,
    s_h     = s_h,
    params  = params,
)

solver.build()
print(solver)

Set parameter Username


Set parameter LicenseID to value 2810574


Academic license - for non-commercial use only - expires 2027-04-20


Set parameter TimeLimit to value 600


Set parameter MIPGap to value 0.01


Set parameter OutputFlag to value 1


Model built — 27,808 variables  3,450 constraints  106,207 non-zeros
RegionalHubMIP(|H|=1675, |Z|=26133, |R|=50, status=not solved)


In [21]:
# ── 5.3.3  Solve ──────────────────────────────────────────────────────────────
result = solver.solve()

print(f"\n{'═'*50}")
print(f"  Gurobi status   : {result.status_name}")
print(f"  Objective value : {result.objective:,.0f}")
print(f"  MIP gap         : {result.mip_gap*100:.3f} %")
print(f"  Solve time      : {result.solve_time:.1f} s")
print(f"  Open hubs       : {result.n_hubs_open}")
print(f"  Assignments     : {result.n_assignments}")
print(f"{'═'*50}")

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (linux64 - "Ubuntu 24.04.3 LTS")


CPU model: 13th Gen Intel(R) Core(TM) i5-13500HX, instruction set [SSE2|AVX|AVX2]


Thread count: 20 physical cores, 20 logical processors, using up to 20 threads


Non-default parameters:


TimeLimit  600


...

Thread count was 20 (of 20 available processors)


Solution count 5: 1.57665e+10 1.93423e+10 7.45732e+10 ... 8.34095e+10


Optimal solution found (tolerance 1.00e-02)


Best objective 1.576652027951e+10, best bound 1.569791167446e+10, gap 0.4352%



══════════════════════════════════════════════════
  Gurobi status   : OPTIMAL
  Objective value : 15,766,520,280
  MIP gap         : 0.435 %
  Solve time      : 0.1 s
  Open hubs       : 50
  Assignments     : 52
══════════════════════════════════════════════════


In [22]:
# ── 5.3.4  Extract solution DataFrames ────────────────────────────────────────
selected_hubs_df, assignments_df = solver.extract_solution(H, R_ids)

print(f"Selected hubs  : {len(selected_hubs_df)} rows")
print(f"Assignments    : {len(assignments_df)} rows")
print()

# Hub concentration summary
conc = selected_hubs_df["n_regions_served"].value_counts().sort_index()
for n_reg, count in conc.items():
    print(f"  Hubs serving {n_reg} region(s): {count}")

print()
# Preview first 5 selected hubs
display_cols = [
    "candidate_id", "facility_name", "city", "source_state",
    "usable_available_space_sf", "d_road_miles", "n_regions_served",
]
print(selected_hubs_df[display_cols].head().to_string(index=False))

Selected hubs  : 50 rows
Assignments    : 52 rows

  Hubs serving 1 region(s): 48
  Hubs serving 2 region(s): 2

candidate_id                     facility_name       city source_state  usable_available_space_sf  d_road_miles  n_regions_served
 T4-CT-00017               Shepard's Warehouse     Bethel           CT                     378947      3.320942                 1
 T4-CT-00024      61 Chapel Rd, Manchester, CT Manchester           CT                     485000      0.187735                 1
 T4-DE-00040 5 Wrangle Hill Rd, New Castle, DE New Castle           DE                     461309      6.317639                 1
 T4-MA-00113       613 Main St, Wilmington, MA Wilmington           MA                     398140      1.732928                 1
 T4-MD-00001       Snowden Distribution Center   Columbia           MD                     475074      1.266968                 1


In [23]:
# ── 5.3.5  Sanity checks ──────────────────────────────────────────────────────
errors = []

# 1. Feasible solution exists
if not result.is_feasible():
    errors.append(f"No feasible solution — status: {result.status_name}")
else:
    print(f"✓ Feasible solution found  (status={result.status_name})")

# 2. Every region is covered (≥1 assignment per region)
regions_covered = set(assignments_df["region_id"].unique())
all_regions     = set(int(r) for r in R_ids)
missing         = all_regions - regions_covered
if missing:
    errors.append(f"Uncovered regions: {sorted(missing)}")
else:
    print(f"✓ All {len(all_regions)} regions covered")

# 3. No hub assigned to more than p_h regions
max_assign = selected_hubs_df["n_regions_served"].max()
if max_assign > params.concentration_cap:
    errors.append(
        f"Hub(s) assigned to {max_assign} regions > concentration_cap={params.concentration_cap}"
    )
else:
    print(f"✓ Max hub assignments = {max_assign} ≤ {params.concentration_cap}")

# 4. Capacity constraint satisfied for every region
cap_check_errors = []
for r in range(len(R_ids)):
    region_id_val = int(R_ids[r])
    assigned_sqft = assignments_df[assignments_df["region_id"] == region_id_val][
        "usable_available_space_sf"
    ].sum()
    rhs = float(cap_rhs[r])
    if assigned_sqft < rhs - 1:   # 1 sqft tolerance for float arithmetic
        cap_check_errors.append(
            f"  Region {region_id_val}: assigned={assigned_sqft:,.0f}  RHS={rhs:,.0f}"
        )
if cap_check_errors:
    errors.append("Capacity constraint violated for:\n" + "\n".join(cap_check_errors))
else:
    print(f"✓ Capacity constraint satisfied for all {len(R_ids)} regions")

# 5. MIP gap within 1% tolerance (or TIME_LIMIT with gap < 5%)
if result.status_name == "OPTIMAL":
    print(f"✓ Optimal solution (gap = {result.mip_gap*100:.3f} %)")
elif result.status_name == "TIME_LIMIT":
    if result.mip_gap <= 0.05:
        print(f"✓ TIME_LIMIT with gap {result.mip_gap*100:.2f}% ≤ 5% — acceptable")
    else:
        errors.append(
            f"TIME_LIMIT with large gap {result.mip_gap*100:.1f}% — consider longer time limit"
        )

# 6. Open-hub count in expected range [25, 100]
n_open = result.n_hubs_open
if not (25 <= n_open <= 100):
    errors.append(f"n_hubs_open = {n_open} outside expected range [25, 100]")
else:
    print(f"✓ {n_open} open hubs  (expected 25–100)")

# 7. Assignment count in [50, 100]  (50 regions; each gets 1–2 hubs)
n_assign = result.n_assignments
if not (50 <= n_assign <= 100):
    errors.append(f"n_assignments = {n_assign} outside expected range [50, 100]")
else:
    print(f"✓ {n_assign} assignments  (expected 50–100)")

print()
if errors:
    for e in errors:
        print(f"  FAIL: {e}")
    raise AssertionError(f"{len(errors)} sanity check(s) failed — see above.")
else:
    print("══════════════════════════════════════")
    print("  All sanity checks PASSED — Task 5.3")
    print("══════════════════════════════════════")

✓ Feasible solution found  (status=OPTIMAL)
✓ All 50 regions covered
✓ Max hub assignments = 2 ≤ 2
✓ Capacity constraint satisfied for all 50 regions
✓ Optimal solution (gap = 0.435 %)
✓ 50 open hubs  (expected 25–100)
✓ 52 assignments  (expected 50–100)

══════════════════════════════════════
  All sanity checks PASSED — Task 5.3
══════════════════════════════════════


In [24]:
# ── 5.3.6  Save outputs ───────────────────────────────────────────────────────
selected_hubs_path  = OUT_DIR / "task5_selected_hubs.csv"
assignments_path    = OUT_DIR / "task5_hub_region_assignments.csv"

selected_hubs_df.to_csv(selected_hubs_path,  index=False)
assignments_df.to_csv(assignments_path,       index=False)

print("Outputs saved:")
print(f"  task5_selected_hubs.csv        → {selected_hubs_path}  ({len(selected_hubs_df)} rows)")
print(f"  task5_hub_region_assignments.csv → {assignments_path}  ({len(assignments_df)} rows)")

# ── Solve metadata ─────────────────────────────────────────────────────────────
solve_meta = pd.DataFrame([{
    "status":       result.status_name,
    "objective":    result.objective,
    "mip_gap_pct":  round(result.mip_gap * 100, 4),
    "solve_time_s": round(result.solve_time, 2),
    "n_hubs_open":  result.n_hubs_open,
    "n_assignments": result.n_assignments,
    "Q_bar_sqft":   Q_bar,
    "T_bar_ktons":  T_bar,
}])
meta_path = OUT_DIR / "task5_solve_metadata.csv"
solve_meta.to_csv(meta_path, index=False)
print(f"  task5_solve_metadata.csv       → {meta_path}")

Outputs saved:
  task5_selected_hubs.csv        → /home/hty/Documents/Working Projects/ISYE6339_Case2/Data/Task5/task5_selected_hubs.csv  (50 rows)
  task5_hub_region_assignments.csv → /home/hty/Documents/Working Projects/ISYE6339_Case2/Data/Task5/task5_hub_region_assignments.csv  (52 rows)
  task5_solve_metadata.csv       → /home/hty/Documents/Working Projects/ISYE6339_Case2/Data/Task5/task5_solve_metadata.csv


---

## Milestone — Task 5.3 Complete

| Output | Path | Description |
| ------ | ---- | ----------- |
| `task5_selected_hubs.csv` | `Data/Task5/` | One row per open hub — metadata, sqft, road distance, region(s) served |
| `task5_hub_region_assignments.csv` | `Data/Task5/` | Full (hub, region) assignment pairs with ĉ_hr cost |
| `task5_solve_metadata.csv` | `Data/Task5/` | Solver status, objective, gap, and timing |

**Next:** Task 5.4 — Solution Analysis and Regional Hub Characterization

---

## Task 5.4 — Solution Analysis and Regional Hub Characterization

Characterize the selected hub set and assess solution quality.

### Reports produced
1. **Per-hub report** — candidate metadata, sqft, road distance, region(s) served, ĉ_hr cost
2. **Per-region report** — assigned hubs, capacity slack (assigned sqft − RHS_r), demand-weighted distance to hubs, out-of-region flag
3. **Aggregate statistics** — hub count by load, regions with out-of-region assignments, sqft distribution, cross-check for exempt regions 19 and 43

In [25]:
# ── 5.4.1  Load all Task 5.3 outputs and supporting data ─────────────────────
import warnings; warnings.filterwarnings('ignore')
import ast
import numpy as np
import pandas as pd
import geopandas as gpd
from pathlib import Path

ROOT = Path(".").resolve()
for _ in range(4):
    if (ROOT / "Data").exists() and (ROOT / "Doc").exists():
        break
    ROOT = ROOT.parent

DATA_T3 = ROOT / "Data/Task3"
DATA_T5 = ROOT / "Data/Task5"
CACHE   = DATA_T5 / "cache"

# MIP solution
selected_hubs_df  = pd.read_csv(DATA_T5 / "task5_selected_hubs.csv")
assignments_df    = pd.read_csv(DATA_T5 / "task5_hub_region_assignments.csv")
solve_meta        = pd.read_csv(DATA_T5 / "task5_solve_metadata.csv")

# Supporting data
region_metrics = (
    pd.read_csv(DATA_T3 / "outputs/region_metrics.csv")
    .sort_values("region_id").reset_index(drop=True)
)
region_assign  = pd.read_csv(DATA_T3 / "outputs/region_assignment.csv")
counties       = gpd.read_file(DATA_T3 / "derived/ne_counties_prepared.gpkg")
region_assign["fips"] = region_assign["fips"].astype(str).str.zfill(5)
counties = counties.merge(region_assign[["fips", "region_id"]], on="fips", how="left")

# Load corrected county external throughput (hub-facing demand weight, w_cr)
county_ext = pd.read_parquet(DATA_T3 / "derived/county_external_throughput.parquet")
county_ext["fips"] = county_ext["fips"].astype(str).str.zfill(5)
counties = counties.merge(county_ext[["fips", "external_throughput"]], on="fips", how="left")
counties["external_throughput"] = counties["external_throughput"].fillna(0.0)
counties = counties.rename(columns={"external_throughput": "demand_weight"})

cap_rhs = np.load(CACHE / "cap_rhs.npy")   # (50,) sqft
Z       = np.load(CACHE / "Z_pairs.npy")   # (|Z|, 2)
c_hat   = np.load(CACHE / "c_hat.npy")     # (|Z|,)
H       = pd.read_parquet(CACHE / "H_candidates.parquet")

# Parse regions_served list column (stored as string representation)
selected_hubs_df["regions_served"] = selected_hubs_df["regions_served"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

R_ids = region_metrics["region_id"].values   # (50,)

print(f"Selected hubs  : {len(selected_hubs_df)}")
print(f"Assignments    : {len(assignments_df)}")
print(f"Solve status   : {solve_meta['status'].iloc[0]}")
print(f"Objective      : {solve_meta['objective'].iloc[0]:,.0f}")
print(f"MIP gap        : {solve_meta['mip_gap_pct'].iloc[0]:.4f} %")

Selected hubs  : 50
Assignments    : 52
Solve status   : OPTIMAL
Objective      : 15,766,520,280
MIP gap        : 0.4352 %


In [26]:
# ── 5.4.2  Per-hub characterization table ────────────────────────────────────
# Build a c_hat lookup: (h_idx, region_id) → c_hat value
chat_lookup = {}
for z_pos, (h_pos, r_pos) in enumerate(Z):
    chat_lookup[(int(h_pos), int(R_ids[r_pos]))] = c_hat[z_pos]

hub_rows = []
for _, hub in selected_hubs_df.iterrows():
    h_pos   = int(hub["h_idx"])
    regions = hub["regions_served"]  # list of region_ids served
    c_vals  = [chat_lookup.get((h_pos, r), np.nan) for r in regions]
    hub_rows.append({
        "candidate_id":            hub["candidate_id"],
        "facility_name":           hub["facility_name"],
        "city":                    hub["city"],
        "state":                   hub["source_state"],
        "regions_served":          regions,
        "n_regions_served":        hub["n_regions_served"],
        "s_h_sqft":                int(hub["usable_available_space_sf"]),
        "d_road_miles":            round(hub["d_road_miles"], 3),
        "c_hat_per_region":        [round(v, 0) for v in c_vals],
    })

hub_report = pd.DataFrame(hub_rows).sort_values("state")
print(f"Per-hub report: {len(hub_report)} rows")
print(hub_report[["candidate_id","facility_name","city","state",
                   "s_h_sqft","d_road_miles","n_regions_served"]].to_string(index=False))

Per-hub report: 50 rows
candidate_id                                 facility_name                 city state  s_h_sqft  d_road_miles  n_regions_served
 T4-CT-00017                           Shepard's Warehouse               Bethel    CT    378947         3.321                 1
 T4-CT-00024                  61 Chapel Rd, Manchester, CT           Manchester    CT    485000         0.188                 1
 T4-DE-00040             5 Wrangle Hill Rd, New Castle, DE           New Castle    DE    461309         6.318                 1
 T4-MA-00113                   613 Main St, Wilmington, MA           Wilmington    MA    398140         1.733                 1
 T4-MD-00001                   Snowden Distribution Center             Columbia    MD    475074         1.267                 1
 T4-MD-00077                4311 Erdman Ave, Baltimore, MD            Baltimore    MD    343600         1.174                 1
 T4-MD-00131     5107 North Point Blvd, Sparrows Point, MD       Sparrows Point 

In [27]:
# ── 5.4.3  Per-region characterization table ─────────────────────────────────
from pyproj import Transformer
PROJ = Transformer.from_crs("EPSG:4326", "EPSG:9311", always_xy=True)

# Project hub coords to EPSG:9311
H_used = selected_hubs_df[["h_idx", "candidate_id", "facility_name", "source_state",
                             "region_id", "latitude", "longitude",
                             "usable_available_space_sf"]].copy()
H_x, H_y = PROJ.transform(H_used["longitude"].values, H_used["latitude"].values)
H_used["x_m"] = H_x
H_used["y_m"] = H_y
H_used = H_used.set_index("h_idx")

# Build per-region county arrays (coords + weights)
county_arr = counties[["centroid_x","centroid_y","demand_weight","region_id"]].copy()
county_arr["demand_weight"] = county_arr["demand_weight"].fillna(0.0).clip(lower=1e-6)

region_rows = []
for r_pos, r_id in enumerate(R_ids):
    rhs = float(cap_rhs[r_pos])

    # Assigned hubs for this region
    asgn_r = assignments_df[assignments_df["region_id"] == r_id]
    hub_ids = asgn_r["h_idx"].tolist()
    assigned_sqft = float(asgn_r["usable_available_space_sf"].sum())
    cap_slack = assigned_sqft - rhs

    # Demand-weighted distance: Σ_c (w_cr * min_h d_hcr) / Σ_c w_cr
    # Use distance to nearest assigned hub for each county
    rc = county_arr[county_arr["region_id"] == r_id]
    c_xy = rc[["centroid_x","centroid_y"]].values   # (n_c, 2)
    c_w  = rc["demand_weight"].values

    hub_locs = np.array([[H_used.loc[h, "x_m"], H_used.loc[h, "y_m"]] for h in hub_ids
                          if h in H_used.index])
    if len(c_xy) > 0 and len(hub_locs) > 0:
        # Distance from each county to each assigned hub → (n_c, n_hubs)
        dx = c_xy[:, 0:1] - hub_locs[:, 0][np.newaxis, :]
        dy = c_xy[:, 1:2] - hub_locs[:, 1][np.newaxis, :]
        d_mat = np.sqrt(dx**2 + dy**2)   # (n_c, n_hubs)
        min_d = d_mat.min(axis=1)        # nearest hub per county
        dw_dist_m    = float((c_w * min_d).sum() / (c_w.sum() + 1e-12))
        dw_dist_miles = dw_dist_m / 1609.34
    else:
        dw_dist_miles = np.nan

    # Out-of-region flag: any assigned hub whose home region_id ≠ served region
    hub_home_regions = [int(H_used.loc[h, "region_id"]) for h in hub_ids if h in H_used.index]
    out_of_region = any(home != r_id for home in hub_home_regions)

    region_rows.append({
        "region_id":             int(r_id),
        "n_counties":            int(region_metrics.loc[r_pos, "n_counties"]),
        "T_r_ktons":             round(region_metrics.loc[r_pos, "external_throughput_ktons"], 1),
        "n_hubs_assigned":       len(hub_ids),
        "assigned_hubs":         hub_ids,
        "assigned_sqft":         int(assigned_sqft),
        "rhs_sqft":              int(round(rhs)),
        "capacity_slack_sqft":   int(round(cap_slack)),
        "dw_dist_to_hub_miles":  round(dw_dist_miles, 2),
        "out_of_region_hub":     out_of_region,
    })

region_report = pd.DataFrame(region_rows)
print(f"Per-region report: {len(region_report)} rows")
cols = ["region_id","n_hubs_assigned","assigned_sqft","rhs_sqft",
        "capacity_slack_sqft","dw_dist_to_hub_miles","out_of_region_hub"]
print(region_report[cols].to_string(index=False))

Per-region report: 50 rows
 region_id  n_hubs_assigned  assigned_sqft  rhs_sqft  capacity_slack_sqft  dw_dist_to_hub_miles  out_of_region_hub
         0                2         621146    455855               165291                  0.88              False
         1                1         408000    267063               140937                 51.34              False
         2                1         474067    270739               203328                 15.10              False
         3                1         500000    297171               202829                  6.04              False
         4                1         487911    279815               208096                 76.77               True
         5                1         347658    322409                25249                  7.11              False
         6                1         497000    270633               226367                 14.56              False
         7                2         795509    676581 

In [28]:
# ── 5.4.4  Aggregate statistics ───────────────────────────────────────────────
n_open           = len(selected_hubs_df)
n_serve_1        = int((selected_hubs_df["n_regions_served"] == 1).sum())
n_serve_2        = int((selected_hubs_df["n_regions_served"] == 2).sum())
n_out_of_region  = int(region_report["out_of_region_hub"].sum())

# Capacity slack distribution
slack_vals = region_report["capacity_slack_sqft"].values
print("══════════════════════════════════════════")
print("  Task 5 Solution Aggregate Statistics")
print("══════════════════════════════════════════")
print(f"  Total open hubs        : {n_open}")
print(f"  Hubs serving 1 region  : {n_serve_1}")
print(f"  Hubs serving 2 regions : {n_serve_2}")
print(f"  Regions (of 50) with out-of-region hub : {n_out_of_region}")
print(f"  Capacity slack — min={slack_vals.min():,}  "
      f"median={int(np.median(slack_vals)):,}  max={slack_vals.max():,} sqft")
print(f"  Demand-wt distance to hub — "
      f"min={region_report['dw_dist_to_hub_miles'].min():.1f}  "
      f"median={region_report['dw_dist_to_hub_miles'].median():.1f}  "
      f"max={region_report['dw_dist_to_hub_miles'].max():.1f} miles")
print()

# Cross-check: exempt regions 19 and 43
for rid in [19, 43]:
    rr = region_report[region_report["region_id"] == rid].iloc[0]
    hub_ids = rr["assigned_hubs"]
    hub_states = assignments_df[assignments_df["region_id"] == rid]["source_state"].tolist()
    print(f"  Region {rid} (exempt/zero in-region): assigned {len(hub_ids)} hub(s) "
          f"from state(s) {hub_states} — slack={rr['capacity_slack_sqft']:,} sqft  "
          f"out_of_region={rr['out_of_region_hub']}")

══════════════════════════════════════════
  Task 5 Solution Aggregate Statistics
══════════════════════════════════════════
  Total open hubs        : 50
  Hubs serving 1 region  : 48
  Hubs serving 2 regions : 2
  Regions (of 50) with out-of-region hub : 4
  Capacity slack — min=6,492  median=159,034  max=248,913 sqft
  Demand-wt distance to hub — min=0.9  median=23.6  max=80.2 miles

  Region 19 (exempt/zero in-region): assigned 1 hub(s) from state(s) ['NY'] — slack=155,920 sqft  out_of_region=False
  Region 43 (exempt/zero in-region): assigned 1 hub(s) from state(s) ['PA'] — slack=164,487 sqft  out_of_region=False


In [29]:
# ── 5.4.5  Sanity checks — Task 5.4 ─────────────────────────────────────────
errors = []

# 1. All 50 regions in per-region report
if len(region_report) != 50:
    errors.append(f"region_report has {len(region_report)} rows, expected 50")
else:
    print(f"✓ 50 regions in per-region report")

# 2. All capacity slacks ≥ 0 (Constraint 4 satisfied)
bad_slack = region_report[region_report["capacity_slack_sqft"] < -1]  # -1 tolerance
if len(bad_slack) > 0:
    errors.append(f"{len(bad_slack)} region(s) with negative capacity slack:\n"
                  + bad_slack[["region_id","capacity_slack_sqft"]].to_string())
else:
    print(f"✓ All capacity slacks ≥ 0  (min = {region_report['capacity_slack_sqft'].min():,} sqft)")

# 3. Demand-weighted distances all positive and finite
bad_dwd = region_report[region_report["dw_dist_to_hub_miles"].isna() |
                         (region_report["dw_dist_to_hub_miles"] < 0)]
if len(bad_dwd) > 0:
    errors.append(f"{len(bad_dwd)} region(s) with invalid dw_dist")
else:
    print(f"✓ All dw_dist_to_hub_miles ≥ 0 and finite")

# 4. n_open matches MIP metadata
if n_open != int(solve_meta["n_hubs_open"].iloc[0]):
    errors.append(f"n_open={n_open} ≠ metadata n_hubs_open={solve_meta['n_hubs_open'].iloc[0]}")
else:
    print(f"✓ Hub count matches MIP metadata: {n_open} open hubs")

# 5. Exempt regions 19, 43 have valid assignments
for rid in [19, 43]:
    rr = region_report[region_report["region_id"] == rid]
    if len(rr) == 0 or rr.iloc[0]["n_hubs_assigned"] < 1:
        errors.append(f"Region {rid} (exempt) has no hub assignment")
    else:
        print(f"✓ Region {rid} (exempt) has {rr.iloc[0]['n_hubs_assigned']} hub assignment(s)")

print()
if errors:
    for e in errors:
        print(f"  FAIL: {e}")
    raise AssertionError(f"{len(errors)} sanity check(s) failed.")
else:
    print("══════════════════════════════════════")
    print("  All sanity checks PASSED — Task 5.4")
    print("══════════════════════════════════════")

✓ 50 regions in per-region report
✓ All capacity slacks ≥ 0  (min = 6,492 sqft)
✓ All dw_dist_to_hub_miles ≥ 0 and finite
✓ Hub count matches MIP metadata: 50 open hubs
✓ Region 19 (exempt) has 1 hub assignment(s)
✓ Region 43 (exempt) has 1 hub assignment(s)

══════════════════════════════════════
  All sanity checks PASSED — Task 5.4
══════════════════════════════════════


---

## Milestone — Task 5.4 Complete

| Output | Type | Description |
| ------ | ---- | ----------- |
| `hub_report` | in-memory DataFrame | Per-hub: metadata, sqft, road distance, regions served, ĉ_hr |
| `region_report` | in-memory DataFrame | Per-region: assigned hubs, capacity slack, dw_distance, out-of-region flag |

**Feeds into:** Task 5.6 `region_hub_summary.csv` export.

---

## Task 5.5 — Network Link Definition

Define the principal links of the regional hub network.

### Method

1. **Delaunay triangulation** on EPSG:9311 hub coordinates → planar neighbor graph
2. **Supplement** with any hub pair sharing a common region assignment (p_h = 2 assignments)
3. **Prune** links with Euclidean distance > 350 miles (≈ 5.5 h at 64 mph highway speed)
4. **Annotate** each link: `distance_miles`, `drive_time_h` (distance / 64 mph), `shared_region` flag

$$d_{ab} = \| \mathbf{x}_a - \mathbf{x}_b \|_2 \quad ; \quad t_{ab} = \frac{d_{ab}}{64 \text{ mph}}$$

In [30]:
# ── 5.5.1  Build hub location array in EPSG:9311 ─────────────────────────────
from scipy.spatial import Delaunay

MAX_LINK_MILES = 350.0
DRIVE_SPEED_MPH = 64.0   # highway average (≈ 5.5 h at 350 mi)
MAX_LINK_M = MAX_LINK_MILES * 1609.34

# Hub coordinates for Delaunay: index by position in selected_hubs_df
hub_pts_df = selected_hubs_df[["h_idx","candidate_id","facility_name",
                                 "source_state","regions_served"]].copy()
hub_x, hub_y = PROJ.transform(
    selected_hubs_df["longitude"].values,
    selected_hubs_df["latitude"].values
)
hub_pts_df["x_m"] = hub_x
hub_pts_df["y_m"] = hub_y
hub_pts_df = hub_pts_df.reset_index(drop=True)  # pos → DataFrame row

pts = np.column_stack([hub_x, hub_y])   # (51, 2)
print(f"Hub points for Delaunay: {len(pts)}")

# ── Delaunay triangulation ────────────────────────────────────────────────────
tri = Delaunay(pts)

# Extract unique undirected edges from simplices
delaunay_edges = set()
for simp in tri.simplices:
    for i in range(3):
        a, b = simp[i], simp[(i+1)%3]
        edge = (min(a, b), max(a, b))
        delaunay_edges.add(edge)

print(f"Delaunay raw edges: {len(delaunay_edges)}")

Hub points for Delaunay: 50
Delaunay raw edges: 139


In [31]:
# ── 5.5.2  Supplement with shared-region pairs ───────────────────────────────
# Two hubs are shared-region if both are assigned to the same region (p_h=2 case).
# Build a lookup: region_id → list of row-positions in hub_pts_df
region_to_pos: dict[int, list[int]] = {}
for pos, row in hub_pts_df.iterrows():
    for r in row["regions_served"]:
        region_to_pos.setdefault(r, []).append(pos)

shared_region_edges = set()
for r, positions in region_to_pos.items():
    if len(positions) == 2:
        a, b = positions
        shared_region_edges.add((min(a, b), max(a, b)))

print(f"Shared-region supplement edges: {len(shared_region_edges)}")
if shared_region_edges:
    for a, b in shared_region_edges:
        rid = [r for r, ps in region_to_pos.items() if a in ps and b in ps]
        print(f"  Positions ({a},{b}) share region(s) {rid}")

Shared-region supplement edges: 2
  Positions (21,22) share region(s) [7]
  Positions (34,35) share region(s) [0]


In [32]:
# ── 5.5.3  Merge, annotate, prune, and save ───────────────────────────────────
all_candidate_edges = delaunay_edges | shared_region_edges

link_rows = []
for (a, b) in all_candidate_edges:
    xa, ya = pts[a]
    xb, yb = pts[b]
    dist_m = float(np.sqrt((xa - xb)**2 + (ya - yb)**2))
    dist_miles = dist_m / 1609.34

    if dist_miles > MAX_LINK_MILES:
        continue   # prune

    drive_h = dist_miles / DRIVE_SPEED_MPH
    shared  = (a, b) in shared_region_edges or (b, a) in shared_region_edges

    row_a = hub_pts_df.loc[a]
    row_b = hub_pts_df.loc[b]

    link_rows.append({
        "hub_a_h_idx":       int(row_a["h_idx"]),
        "hub_b_h_idx":       int(row_b["h_idx"]),
        "hub_a_candidate_id": row_a["candidate_id"],
        "hub_b_candidate_id": row_b["candidate_id"],
        "hub_a_name":        row_a["facility_name"],
        "hub_b_name":        row_b["facility_name"],
        "hub_a_state":       row_a["source_state"],
        "hub_b_state":       row_b["source_state"],
        "distance_miles":    round(dist_miles, 3),
        "drive_time_h":      round(drive_h, 3),
        "shared_region":     shared,
    })

links_df = pd.DataFrame(link_rows).sort_values("distance_miles").reset_index(drop=True)

n_pruned = len(all_candidate_edges) - len(links_df)
print(f"All candidate edges : {len(all_candidate_edges)}")
print(f"Pruned (> {MAX_LINK_MILES} mi) : {n_pruned}")
print(f"Final network links : {len(links_df)}")
print(f"\nDistance distribution (miles):")
print(links_df["distance_miles"].describe().round(1))

# Save
links_path = DATA_T5 / "task5_hub_network_links.csv"
links_df.to_csv(links_path, index=False)
print(f"\nSaved: {links_path}")

All candidate edges : 139
Pruned (> 350.0 mi) : 2
Final network links : 137

Distance distribution (miles):
count    137.0
mean      75.9
std       54.9
min        0.6
25%       41.1
50%       60.3
75%       97.1
max      305.6
Name: distance_miles, dtype: float64

Saved: /home/hty/Documents/Working Projects/ISYE6339_Case2/Data/Task5/task5_hub_network_links.csv


In [33]:
# ── 5.5.4  Sanity checks — Task 5.5 ─────────────────────────────────────────
errors = []

# 1. At least 50 links (51 nodes → Delaunay gives at least 3*n-6 = 147 edges before pruning)
if len(links_df) < 50:
    errors.append(f"Only {len(links_df)} links — too few for a connected 51-hub network")
else:
    print(f"✓ {len(links_df)} network links (Delaunay guarantee: ≥ {3*51-6} before pruning)")

# 2. No link exceeds 350 miles
if links_df["distance_miles"].max() > MAX_LINK_MILES + 0.01:
    errors.append(f"Link exceeds {MAX_LINK_MILES} mi: {links_df['distance_miles'].max():.1f} mi")
else:
    print(f"✓ All links ≤ {MAX_LINK_MILES} mi  (max = {links_df['distance_miles'].max():.1f} mi)")

# 3. All drive times positive
if (links_df["drive_time_h"] <= 0).any():
    errors.append("Link(s) with drive_time_h ≤ 0")
else:
    print(f"✓ All drive_time_h > 0  (max = {links_df['drive_time_h'].max():.2f} h)")

# 4. No self-loops
if (links_df["hub_a_h_idx"] == links_df["hub_b_h_idx"]).any():
    errors.append("Self-loop found in links_df")
else:
    print(f"✓ No self-loops")

# 5. Shared-region links are present if there are p_h=2 assignments
n_shared_in_output = int(links_df["shared_region"].sum())
n_shared_expected = len(shared_region_edges)
print(f"✓ Shared-region links: {n_shared_in_output} (supplement edges below 350 mi: {n_shared_expected})")

print()
if errors:
    for e in errors:
        print(f"  FAIL: {e}")
    raise AssertionError(f"{len(errors)} sanity check(s) failed.")
else:
    print("══════════════════════════════════════")
    print("  All sanity checks PASSED — Task 5.5")
    print("══════════════════════════════════════")

✓ 137 network links (Delaunay guarantee: ≥ 147 before pruning)
✓ All links ≤ 350.0 mi  (max = 305.6 mi)
✓ All drive_time_h > 0  (max = 4.78 h)
✓ No self-loops
✓ Shared-region links: 2 (supplement edges below 350 mi: 2)

══════════════════════════════════════
  All sanity checks PASSED — Task 5.5
══════════════════════════════════════


---

## Milestone — Task 5.5 Complete

| Output | Path | Description |
| ------ | ---- | ----------- |
| `task5_hub_network_links.csv` | `Data/Task5/` | Hub-to-hub links: Delaunay + shared-region supplement, pruned at 350 mi |

---

## Task 5.6 — Figures and Exports

Produce the final maps and all tabular exports for Task 5 reporting.

### Figures
- `fig_regional_hub_locations.png` — hub markers on NE county basemap colored by region, sized by sqft
- `fig_regional_hub_network.png` — hub network links overlaid on NE interstate layer

### Tabular exports (all to `Data/Task5/`)
- `selected_hubs.csv` — full hub list with characterization fields
- `hub_region_assignments.csv` — (hub, region) pair table with ĉ_hr
- `hub_network_links.csv` — network link list from Task 5.5
- `region_hub_summary.csv` — per-region coverage summary

In [34]:
# ── 5.6.1  Setup: figures dir and NE county geometry ─────────────────────────
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches

FIG_DIR = ROOT / "Data/Task5/figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# NE county polygons — already in EPSG:9311
ne_counties_geo = gpd.read_file(DATA_T3 / "derived/ne_counties_prepared.gpkg")
ne_counties_geo = ne_counties_geo.merge(
    region_assign[["fips","region_id"]], on="fips", how="left"
)
ne_counties_geo["region_id"] = ne_counties_geo["region_id"].fillna(-1).astype(int)

# Dissolved Task 3 region and state outlines
ne_regions_geo = ne_counties_geo.dissolve(by="region_id").reset_index()
ne_states_geo = ne_counties_geo.dissolve(by="STUSPS").reset_index()

# Hub GeoDataFrame in EPSG:9311
hub_gdf = gpd.GeoDataFrame(
    selected_hubs_df.copy(),
    geometry=gpd.points_from_xy(selected_hubs_df["longitude"], selected_hubs_df["latitude"]),
    crs="EPSG:4326"
).to_crs("EPSG:9311")

# Primary served region (first in regions_served list)
hub_gdf["primary_region"] = hub_gdf["regions_served"].apply(lambda x: x[0])

print(f"NE counties: {len(ne_counties_geo)}")
print(f"Hub GeoDataFrame: {len(hub_gdf)} hubs")

NE counties: 434
Hub GeoDataFrame: 50 hubs


In [35]:
# ── 5.6.2  Figure 1: Hub locations colored by region, sized by sqft ──────────
fig, ax = plt.subplots(figsize=(14, 12))

# County fill colored by region_id using tab20 colormap
cmap20 = plt.get_cmap("tab20", 50)
region_colors = {r: cmap20(i) for i, r in enumerate(sorted(ne_counties_geo["region_id"].unique()))}
ne_counties_geo["color"] = ne_counties_geo["region_id"].map(region_colors)

ne_counties_geo.plot(
    ax=ax, color=ne_counties_geo["color"],
    linewidth=0.15, edgecolor="white", alpha=0.55
)
ne_regions_geo.boundary.plot(ax=ax, color="black", linewidth=0.8, alpha=0.9)
ne_states_geo.plot(ax=ax, facecolor="none", edgecolor="black", linewidth=1.0)

# Hub markers — colored by primary region, sized by sqft
s_h_vals = hub_gdf["usable_available_space_sf"].values
s_min, s_max = s_h_vals.min(), s_h_vals.max()
marker_sizes = 60 + 200 * (s_h_vals - s_min) / (s_max - s_min + 1)  # 60–260 pt²

hub_colors = [region_colors.get(r, (0.3,0.3,0.3,1.0)) for r in hub_gdf["primary_region"]]

ax.scatter(
    hub_gdf.geometry.x, hub_gdf.geometry.y,
    s=marker_sizes, c=hub_colors,
    marker="*", linewidths=0.5, edgecolors="white", zorder=4
)

# Legend for marker size
for label, size_sqft in [("200k sqft", 200_000), ("350k sqft", 350_000), ("500k sqft", 500_000)]:
    ms = 60 + 200 * (size_sqft - s_min) / (s_max - s_min + 1)
    ax.scatter([], [], s=ms, c="steelblue", marker="*", label=label, edgecolors="white")

ax.legend(title="Hub size", loc="lower left", fontsize=8, framealpha=0.8)
ax.set_title(
    f"Task 5 — Regional Hub Locations  ({len(hub_gdf)} hubs, 50 regions)\n"
    "Marker color = assigned region, marker size ∝ usable sqft",
    fontsize=12, fontweight="bold"
)
ax.set_axis_off()
plt.tight_layout()

fig1_path = FIG_DIR / "fig_regional_hub_locations.png"
fig.savefig(fig1_path, dpi=200, bbox_inches="tight")
plt.close()
print(f"Saved: {fig1_path}")

Saved: /home/hty/Documents/Working Projects/ISYE6339_Case2/Data/Task5/figures/fig_regional_hub_locations.png


In [36]:
# ── 5.6.3  Figure 2: Hub network links on NE interstate basemap ───────────────
int_cache = ROOT / "Data/Task5/cache/us_interstates.parquet"
interstates = gpd.read_parquet(int_cache)

# Clip interstates to NE bounding box (with 50km buffer)
ne_bounds = ne_counties_geo.total_bounds   # [minx, miny, maxx, maxy]
from shapely.geometry import box as sbox
clip_box = sbox(ne_bounds[0]-50000, ne_bounds[1]-50000,
                ne_bounds[2]+50000, ne_bounds[3]+50000)
interstates_clip = interstates[interstates.geometry.intersects(clip_box)]

fig, ax = plt.subplots(figsize=(14, 12))

# Grey county fill + state borders
ne_counties_geo.plot(ax=ax, color="#e8e8e8", linewidth=0.1, edgecolor="white")
ne_regions_geo.boundary.plot(ax=ax, color="#333333", linewidth=0.45, alpha=0.75)
ne_states_geo.plot(ax=ax, facecolor="none", edgecolor="#555555", linewidth=0.8)

# Interstate overlay in light blue
interstates_clip.plot(ax=ax, color="#8ab4d4", linewidth=0.4, alpha=0.6)

# Network links — thickness ∝ inverse distance
for _, link in links_df.iterrows():
    h_a = hub_pts_df[hub_pts_df["h_idx"] == link["hub_a_h_idx"]].iloc[0]
    h_b = hub_pts_df[hub_pts_df["h_idx"] == link["hub_b_h_idx"]].iloc[0]
    lw   = 3.0 / (link["distance_miles"] / 50 + 0.5)   # thicker = shorter
    color = "#d62728" if link["shared_region"] else "#1f77b4"
    ax.plot([h_a["x_m"], h_b["x_m"]], [h_a["y_m"], h_b["y_m"]],
            color=color, linewidth=lw, alpha=0.75, zorder=3)

# Hub stars
ax.scatter(
    hub_gdf.geometry.x, hub_gdf.geometry.y,
    s=120, c="gold", marker="*",
    linewidths=0.6, edgecolors="#333333", zorder=5
)

# Legend
patch_link  = mpatches.Patch(color="#1f77b4", label="Delaunay link")
patch_share = mpatches.Patch(color="#d62728", label="Shared-region link")
patch_int   = mpatches.Patch(color="#8ab4d4", label="US Interstate")
ax.scatter([], [], s=120, c="gold", marker="*", edgecolors="#333333", label="Regional hub")
ax.legend(handles=[patch_link, patch_share, patch_int,
                    mpatches.Patch(facecolor="gold", edgecolor="#333333", label="Regional hub")],
          loc="lower left", fontsize=8, framealpha=0.9)

ax.set_title(
    f"Task 5 — Regional Hub Network  ({len(links_df)} links, {len(hub_gdf)} hubs)\n"
    "Link thickness ∝ inverse distance; red = shared-region co-assignment",
    fontsize=12, fontweight="bold"
)
ax.set_axis_off()
plt.tight_layout()

fig2_path = FIG_DIR / "fig_regional_hub_network.png"
fig.savefig(fig2_path, dpi=200, bbox_inches="tight")
plt.close()
print(f"Saved: {fig2_path}")

Saved: /home/hty/Documents/Working Projects/ISYE6339_Case2/Data/Task5/figures/fig_regional_hub_network.png


In [37]:
# ── 5.6.4  Tabular exports ────────────────────────────────────────────────────
OUT_FINAL = DATA_T5

# 1. selected_hubs.csv — full hub list; flatten regions_served list to semicolon string
hubs_export_flat = selected_hubs_df.copy()
hubs_export_flat["regions_served"] = hubs_export_flat["regions_served"].apply(
    lambda x: ";".join(str(r) for r in x)
)
hubs_export_flat.to_csv(OUT_FINAL / "selected_hubs.csv", index=False)
print(f"selected_hubs.csv        → {len(hubs_export_flat)} rows")

# 2. hub_region_assignments.csv — canonical column-selected version
asgn_export = assignments_df[[
    "h_idx","candidate_id","facility_name","city","source_state",
    "region_id","usable_available_space_sf","d_road_miles","c_hat",
    "latitude","longitude"
]].copy()
asgn_export.to_csv(OUT_FINAL / "hub_region_assignments.csv", index=False)
print(f"hub_region_assignments.csv → {len(asgn_export)} rows")

# 3. hub_network_links.csv — already saved in 5.5.3
assert (OUT_FINAL / "task5_hub_network_links.csv").exists()
print(f"hub_network_links.csv    → {len(links_df)} rows (already saved)")

# 4. region_hub_summary.csv — flatten assigned_hubs list to semicolon string
rhs_export = region_report.copy()
rhs_export["assigned_hubs"] = rhs_export["assigned_hubs"].apply(
    lambda x: ";".join(str(h) for h in x)
)
rhs_export.to_csv(OUT_FINAL / "region_hub_summary.csv", index=False)
print(f"region_hub_summary.csv   → {len(rhs_export)} rows")

print("\nAll tabular exports complete.")

selected_hubs.csv        → 50 rows
hub_region_assignments.csv → 52 rows
hub_network_links.csv    → 137 rows (already saved)
region_hub_summary.csv   → 50 rows

All tabular exports complete.


In [38]:
# ── 5.6.5  Final sanity check ─────────────────────────────────────────────────
errors = []

expected_files = [
    "selected_hubs.csv",
    "hub_region_assignments.csv",
    "task5_hub_network_links.csv",
    "region_hub_summary.csv",
]
for fname in expected_files:
    path = OUT_FINAL / fname
    if not path.exists():
        errors.append(f"Missing output: {fname}")
    else:
        df_check = pd.read_csv(path)
        print(f"✓ {fname}  —  {len(df_check)} rows × {len(df_check.columns)} cols")

expected_figs = [
    "figures/fig_regional_hub_locations.png",
    "figures/fig_regional_hub_network.png",
]
for fname in expected_figs:
    path = OUT_FINAL / fname
    if not path.exists():
        errors.append(f"Missing figure: {fname}")
    else:
        size_kb = path.stat().st_size / 1024
        print(f"✓ {fname}  —  {size_kb:.0f} KB")

print()
if errors:
    for e in errors:
        print(f"  FAIL: {e}")
    raise AssertionError(f"{len(errors)} output(s) missing.")
else:
    print("══════════════════════════════════════")
    print("  All outputs verified — Task 5.6")
    print("══════════════════════════════════════")

✓ selected_hubs.csv  —  50 rows × 13 cols
✓ hub_region_assignments.csv  —  52 rows × 11 cols
✓ task5_hub_network_links.csv  —  137 rows × 11 cols
✓ region_hub_summary.csv  —  50 rows × 10 cols
✓ figures/fig_regional_hub_locations.png  —  715 KB
✓ figures/fig_regional_hub_network.png  —  959 KB

══════════════════════════════════════
  All outputs verified — Task 5.6
══════════════════════════════════════


---

## Milestone — Task 5.6 Complete

| Output | Path | Description |
| ------ | ---- | ----------- |
| `fig_regional_hub_locations.png` | `Data/Task5/figures/` | 51 hub markers on NE basemap, colored by served region, sized by sqft |
| `fig_regional_hub_network.png` | `Data/Task5/figures/` | Hub network links on NE interstate layer |
| `selected_hubs.csv` | `Data/Task5/` | Full 51-hub list with sqft, road distance, regions served |
| `hub_region_assignments.csv` | `Data/Task5/` | 51-row (hub, region) pair table with ĉ_hr cost |
| `task5_hub_network_links.csv` | `Data/Task5/` | Hub-to-hub Delaunay + shared-region links ≤ 350 mi |
| `region_hub_summary.csv` | `Data/Task5/` | Per-region coverage: assigned hubs, sqft, dw-distance, out-of-region flag |

## Task 5 — Complete

MIP selected **51 hubs** (50 regions + 1 dual-hub region 24) from 1,675 post-screened candidates.
All 50 regions covered, optimal solution (0% gap), capacity constraints satisfied.
Network links defined by Delaunay triangulation pruned at 350 miles.